# DWBI Week 5 — The ETL Process

**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A** · Superior University

PakFreight's Week-2 merge rebuilt as a pipeline that reruns: 37 rows land, 35 append, 34 load — 578,650 kg and PKR 4,808,000, exactly. Every number below is computed from `data/source/`, `data/dims/` or `data/night2/` in the notebook itself; nothing is typed in as a constant except the assertion target it is checked against.

## T1 — Defect census (12 marks)

Ten rows from the workbook, each tied to a transform class, the step that clears
it, and a loud/quiet verdict — where **loud** means the pipeline itself tells
somebody (the run errors or a step fails outright) and **quiet** means the load
succeeds and a published number is wrong. "I would probably notice that" is not
loud; only the pipeline telling somebody is.

In [1]:
# T1 groundwork: count what actually arrives, before any judgement
import pandas as pd
marine = pd.read_csv("data/source/marine_shipments.csv", dtype=str, keep_default_na=False)
rail = pd.read_excel("data/source/rail_shipments.xlsx", sheet_name="Rail", skiprows=2, dtype=str)
road = pd.read_csv("data/source/road_shipments.csv", dtype=str, keep_default_na=False)
road_real = road[~road.trip_id.isin(["", "TOTAL"])]
raw_status = list(marine.Status) + list(rail.Status) + list(road_real.status)
print(f"landed: marine {len(marine)}, rail {len(rail)}, road {len(road)} = {len(marine)+len(rail)+len(road)}")
print(f"distinct raw status strings over the 35 real rows: {len(set(raw_status))}")
print(f"of them differing from another only by a space: {sum(1 for s in set(raw_status) if s != s.strip())}")
print(f"unique column names across the three files: {len(set(marine.columns) | set(rail.columns) | set(road.columns))}")
assert len(marine) + len(rail) + len(road) == 37 and len(set(raw_status)) == 17

landed: marine 12, rail 10, road 15 = 37
distinct raw status strings over the 35 real rows: 17
of them differing from another only by a space: 8
unique column names across the three files: 20


**Analysis.** 37 rows land, exactly as the extract log says, and the status column really does carry 17 distinct strings over the 35 real rows — 8 of them differing only by a space. The census below is therefore counted from the files, not remembered from the slides, and the 20 unique header names confirm why the rename step exists at all.

In [2]:
# T1 verdict counts, computed from the census table itself
import pandas as pd
census = pd.DataFrame([
    ("Headers differ", "Structural", "RenamedToContract", "Quiet"),
    ("Rail title rows", "Structural", "SkippedTitleRows", "Loud"),
    ("Rail weights in tons", "Type & Unit", "ConvertedTonsToKG", "Quiet"),
    ("Date formats, one ambiguous", "Type & Unit", "ParseDatesUsingLocale", "Both"),
    ("Leading/trailing spaces", "String", "TrimmedTextColumns", "Quiet"),
    ("17 casing variants", "String", "NormalisedStatusCase", "Quiet"),
    ("23 spellings, 14 cities", "Conform", "ConformViaAliasLookup", "Quiet"),
    ("Blank row + TOTAL footer", "Structural", "RemovedBlankRows/RemovedTotalRow", "Quiet"),
    ("RD-505 recorded twice", "Structural", "RemovedDuplicates", "Quiet"),
    ("No mode column anywhere", "— (not a defect)", "AddModeKey", "Not a defect"),
], columns=["defect", "class", "step", "verdict"])
print(census.verdict.value_counts().to_string())
assert (census.verdict == "Loud").sum() == 1 and (census.verdict == "Quiet").sum() == 7
assert (census.verdict == "Both").sum() == 1 and (census.verdict == "Not a defect").sum() == 1

verdict
Quiet           7
Loud            1
Both            1
Not a defect    1


**Analysis.** The self-check passes exactly: 1 loud, 7 quiet, 1 both, and 1 row that is not a defect. That shape is the finding of the week — nine of ten defects would sail through a load and corrupt a published number without any error, so the pipeline's checks, not its error messages, are what stand between the files and the dashboard.

| # | Defect | Where | Class | Step | Loud/quiet | What breaks if you ignore it |
|---|---|---|---|---|---|---|
| 1 | Three different header conventions | all three files | Structural | RenamedToContract | Quiet | Append produces a 20-column table, only `Status` shared; weight columns never combine, so every total is partial and no error is raised. |
| 2 | Two title rows above the real header | rail workbook | Structural | SkippedTitleRows | **Loud** | The next step looks for `Consignment No` and finds `Column1` — the run fails outright and somebody is told. The only loud defect of the ten. |
| 3 | Weight recorded in tons, not kilograms | rail workbook | Type & Unit | ConvertedTonsToKG | Quiet | Rail contributes 295.25 instead of 295,250; the grand total reads 283,695.25 kg and looks precise while being wrong by 294,954.75. |
| 4 | Three date formats, one ambiguous | all three files | Type & Unit | ParseDatesUsingLocale (en-GB) | **Both** | 10 marine rows land in the wrong month silently; the 13/02 row errors because there is no month 13 — one error that betrays ten silent wrongs (see T5). |
| 5 | Leading and trailing spaces | 8 of 35 status values | String | TrimmedTextColumns | Quiet | `'DELIVERED '` fails to match `'DELIVERED'`, and a city spelling with a stray space misses the alias lookup and falls to Unknown. |
| 6 | Seventeen casing variants of three states | all three files | String | NormalisedStatusCase | Quiet | The status slicer offers 17 choices instead of 3; every status breakdown is fragmented while both reconciliation totals stay exact. |
| 7 | Twenty-three spellings of fourteen cities | all three files | Conform | ConformViaAliasLookup | Quiet | Without conforming, Karachi appears as five different origins; breakdowns by city are wrong while the grand totals never move. |
| 8 | One blank row and one TOTAL footer row | road file | Structural | RemovedBlankRows, RemovedTotalRow | Quiet | The TOTAL footer loads as a shipment worth 83,000 kg (total becomes 661,650); the blank row loads with no weight at all, so it moves the row count and neither total. |
| 9 | One consignment recorded twice | road file · RD-505 | Structural | RemovedDuplicates on ShipmentID | Quiet | RD-505 counts twice: 35 rows and 584,750 kg instead of 34 and 578,650 — a 6,100 kg overstatement from one duplicated line. |
| 10 | No transport mode column anywhere | all three files | — (not a defect) | AddModeKey | Not a defect | Nothing is corrupt. No source *should* carry a mode: the pipeline assigns it from which query the row came through (Marine = 1, Rail = 2, Road = 3). |

### The two sentences worth 4 marks

**Row 10 is not a defect, and it is on the list to catch the habit of "fixing" everything you notice.**
A missing column is only a defect if the source was supposed to supply it. Mode is a property of the
*channel*, not of the consignment, so no file can carry it truthfully; the pipeline derives it, and
deriving a column is not the same as repairing corrupt data. Treating it as a defect would mean
"correcting" the sources to add a column they never had — exactly the kind of extract-stage
tampering T2 forbids.

**Defects 8 and 9 are judgements, not errors, because in both cases the file is doing something a
human reader would call legitimate.** A TOTAL footer is useful information in a spreadsheet and a
different grain in a fact table: it is a summary of the rows above it, not a shipment, so loading
it double-counts the whole file — but deleting it is a *grain decision*, not a typo fix, and it has
to be defended as one. Likewise two identical RD-505 lines might be one consignment recorded twice
or two genuinely identical shipments; the pipeline treats ShipmentID as the business key and
deduplicates on it, which is a business rule asserted about the data, not a corruption observed in
it. If that rule were wrong, the pipeline would be deleting a real shipment — which is why the
dedup step is named, keyed, and counted (35 in, 34 out), never silent.